# mini-beatrix-3 + the lookup arm — content-addressed attention added to the released all-splat trunk

**Why.** The byte battery read the released mini-beatrix-3 bit by bit. Given the first byte of a planted value it continues
the copy (.88-.98 near, fading past 1,024 bytes), but it never makes the jump from a cue to the value behind text (the
first byte at .00-.06 at every distance), it loses order inside a repeated digit string (position 12 at .23), and with
four records it answers by position, never by key. The softmax twin fetches the first byte at .9-1.0 to the end of its
context, and the old hybrid's softmax blocks did too. The missing step is content-addressed lookup.

**What this notebook does.** It adds a detachable softmax attention arm after each block in `SITES` of the trained v3:
the same attention module the softmax crafts use (QK-norm from birth, fp16 flash inside the fp32 attention block: the
guards that held the twin), behind its own LayerNorm, with its output projection at zero. At step 0 the model is v3
exactly; loading the released checkpoint leaves only the arm's keys fresh. Training touches the arm alone (the trunk is
frozen, under pure Adam at the house rate) on the far-recall mix (fineweb 95% + planted records with their questions at
64-2,400 bytes, in single, locker and update forms) for `STEPS` steps of the recipe's 262,144-token step. The craft is
its own prefix on the training repo (`mini-beatrix-3-la2-6-12-20` for the default sites); checkpoints ship as it goes;
C4 also ships the arm alone (a few million parameters) so v3 can carry it or drop it.

**The pass marks** (the battery and the probes, run on the shipped checkpoints): the first byte of a planted value at
1,024 and 2,048 bytes at the twin's level (≥ .9), the four-locker lookup above chance (.25), the repeated digit string's
position 12 toward .9, held-out bits per byte not worse than v3's. Decode with the arm is not wired yet (forward only).

In [ ]:
# C1 — pinned install. RESTART RUNTIME if the version line below changes.
import os
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ.setdefault('OMP_NUM_THREADS', '16')
%pip install -q "datasets>=5.0.0" "geolip-alephllm[train] @ git+https://github.com/AbstractEyes/alephllm@v0.10.15"
import geolip.alephllm as A
print('geolip.alephllm', A.__version__)
assert tuple(int(x) for x in A.__version__.split('.')) >= (0, 10, 15), (
    'RESTART REQUIRED: runtime still holds the old package — Runtime > Restart, then rerun from C1')

from geolip.alephllm.presets import make_v3_lookup_arm_preset

SITES = (2, 6, 12, 20)        # blocks that get an arm (the arm reads the block's output); two early for the circuit, two deeper
STEPS = 4000                  # arm steps at 262,144 tokens each (~2.1 B bytes of the far-recall mix)
MAX_HOURS = None              # None: STEPS at the pace C2 measures (+5%); or a cap in hours
MICRO_BATCH = 8               # 8 x 8 on an 80 GB card; a smaller card: 4 (x 16) or 2 (x 32); the step stays 262,144 tokens

PRESET = make_v3_lookup_arm_preset(SITES, STEPS, micro_batch=MICRO_BATCH, grad_accum=64 // MICRO_BATCH)
cfg, tc = PRESET.model, PRESET.train
print(f"{cfg.name}: 32 splat blocks + arms at {list(cfg.lookup_arm_sites)} | arm switches: qk_norm {cfg.qk_norm!r} attn_fp32 {cfg.attn_fp32} kernel {cfg.attn_kernel!r} | "
      f"trains {list(tc.arm_params)} under Adam {tc.adam_lr} | {tc.micro_batch} x {tc.grad_accum} | phases {[ (ph['name'], ph['dataset']) for ph in PRESET.curriculum]}")

In [ ]:
# C2 (P) — PREFLIGHT: build the armed trunk, load the released weights (only the arm keys fresh), the birth identity on a real
# batch, the arm-only bench (sets MAX_HOURS), VRAM gate. No training, no hub writes.
import time, torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from geolip.alephllm.model.alephlm import AlephLM
from geolip.alephllm.data.tokenizers import build_tokenizer
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.train.precision import autocast
from geolip.alephllm.train.optim import build_optimizers, freeze_except
def hf_token():
    import os
    t = os.environ.get('HF_TOKEN')
    if not t:
        from google.colab import userdata
        t = userdata.get('HF_TOKEN')
    assert t, 'no HF token: set HF_TOKEN in the environment (a pod) or in Colab secrets'
    return t

src = PRESET.init_from
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
torch.manual_seed(tc.seed)
model = AlephLM(cfg).cuda()
ck = hf_hub_download(src['repo'], f"{src['prefix']}/{src['path']}", token=hf_token())
sd = {k: v.float() for k, v in load_file(ck).items()}
res = model.load_state_dict(sd, strict=False)
arm_keys = {k for k in model.state_dict() if k.startswith(('lookup_arms.', 'lookup_norms.'))}
assert not res.unexpected_keys and set(res.missing_keys) == arm_keys, (res.missing_keys[:4], list(res.unexpected_keys)[:4])
n_arm = sum(p.numel() for n, p in model.named_parameters() if n.startswith(tuple(tc.arm_params)))
print(f"{model.param_count()/1e6:.1f}M params, the arm {n_arm/1e6:.2f}M in {len(arm_keys)} tensors (fresh); the trunk from {src['prefix']}/{src['path']}")

# THE BIRTH IDENTITY: the same batch with the arms detached and attached -> identical loss (the output projections are zero)
tok = build_tokenizer(cfg.tokenizer)
x = build_stream('fineweb-edu', tok, cfg.context, 2, seed=tc.seed + 99).next_batch().cuda()
model.eval()
with torch.no_grad(), autocast('cuda', tc.precision):
    arms = model.lookup_arms; model.lookup_arms = torch.nn.ModuleDict()
    l_bare = float(model(x[:, :-1], targets=x[:, 1:]).loss)
    model.lookup_arms = arms
    l_armed = float(model(x[:, :-1], targets=x[:, 1:]).loss)
print(f"birth identity: v3 {l_bare:.5f} nats, v3 + arms {l_armed:.5f} ({l_armed - l_bare:+.2e}); {l_bare / 0.6931:.4f} bits/byte")
assert abs(l_armed - l_bare) < 1e-4, 'the arm is not null at birth'
model.train(); del x

# THE BENCH: arm-only training steps (the trunk frozen) at the preset's micro-batch -> the pace -> MAX_HOURS
n_train, n_all = freeze_except(model, tc.arm_params)
muon, adam = build_optimizers(model, tc.muon_lr, tc.muon_momentum, tc.adam_lr, adam_prefixes=tc.arm_params)
xb = torch.randint(0, 255, (tc.micro_batch, cfg.context), device='cuda')
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
for i in range(3):
    with autocast('cuda', tc.precision):
        out = model(xb, targets=xb)
    out.loss.backward(); adam.zero_grad(set_to_none=True); muon.zero_grad(set_to_none=True)
torch.cuda.synchronize()
sps = (time.time() - t0) / 3 * tc.grad_accum
vram = torch.cuda.max_memory_allocated() / 2**30
total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'bench on {torch.cuda.get_device_name(0)} ({total:.0f} GB), arm-only backward through the frozen trunk: {sps:.2f} s/step at {tc.micro_batch} x {tc.grad_accum} · peak {vram:.1f} GB · {n_train:,} of {n_all:,} parameters train')
assert vram < 0.85 * total, f'over the card budget ({total:.0f} GB): lower MICRO (the step stays 262,144 tokens)'
if MAX_HOURS is None:
    MAX_HOURS = STEPS * sps / 3600 * 1.05
print(f'MAX_HOURS = {MAX_HOURS:.2f} ({STEPS} steps + 5%)')
del model, out, sd, xb, muon, adam; torch.cuda.empty_cache()
print('PREFLIGHT PASS — C3 is armed')

In [ ]:
# C3 (T) — THE SESSION: the weights-only start from the released v3 (the trainer: the arm keys fresh, the cursor on the arm's
# first step, the trunk frozen, the arm under Adam), training to the arm phase's end (STEPS) or MAX_HOURS, a checkpoint every
# 1,000 steps and at the end, the end report. A later session on this craft is a normal resume.
import gc, json, time, torch
from geolip.alephllm import prepare
from geolip.alephllm.train import probes
from geolip.alephllm.train.instruments import model_census, toggle_ledger, special_token_gauge
def hf_token():
    import os
    t = os.environ.get('HF_TOKEN')
    if not t:
        from google.colab import userdata
        t = userdata.get('HF_TOKEN')
    assert t, 'no HF token: set HF_TOKEN in the environment (a pod) or in Colab secrets'
    return t

run = prepare(PRESET, hf_token=hf_token())
if run.manifest.init_from:
    print('start:', json.dumps({k: v for k, v in run.manifest.init_from.items() if k != 'qk_gains'}, indent=1))

def end_report(run, tag):
    m, tok, dev = run.raw_model, run.tokenizer, run.device
    m.eval()
    with torch.no_grad():
        pr = probes.run_all(m, tok, dev)
        sample = torch.randint(0, 255, (2, min(1024, m.cfg.context)), device=dev)
        census = model_census(m, sample)
        ledger = toggle_ledger(m, run._val())
        sp = special_token_gauge(m, run._val())
    rep = {'tag': tag, 'step': run.step, 'tokens': run.manifest.tokens_seen, 'craft': run.cfg.name,
           'sites': list(run.cfg.lookup_arm_sites), 'arm_params': list(run.tc.arm_params),
           'init_from': {k: v for k, v in (run.manifest.init_from or {}).items() if k != 'qk_gains'},
           'probes': pr, 'census_flags': census.get('flags'), 'ledger': ledger, 'special': sp}
    run.hub.upload_bytes(json.dumps(rep, default=float).encode(), f'reports/lookup_arm/{tag}_step{run.step}.json')
    print(probes.report(pr)); m.train()
    return rep

t_end = time.time() + MAX_HOURS * 3600
while time.time() < t_end:
    hours_left = (t_end - time.time()) / 3600
    if hours_left < 0.05: break
    run.train(max_hours=hours_left, stop_at_boundary=True)
    if getattr(run, '_interrupted', False):
        print('manual stop - no auto-resume; resume state is on the hub'); break
    if run.manifest.current_phase() is None:
        print('the arm phase is complete'); break
    if getattr(run, '_last_boundary', None):
        break
end_report(run, 'arm_end')
print('session over — resume state on the hub')

In [ ]:
# C4 — the detachable arm: the latest checkpoint's lookup_arms.* / lookup_norms.* tensors alone, shipped beside it
# (v3 + this file = the armed model; v3 alone = the release, untouched).
import io, torch
from huggingface_hub import HfApi, hf_hub_download
from safetensors.torch import load_file, save
from geolip.alephllm.presets import TRAINING_REPO
def hf_token():
    import os
    t = os.environ.get('HF_TOKEN')
    if not t:
        from google.colab import userdata
        t = userdata.get('HF_TOKEN')
    assert t, 'no HF token: set HF_TOKEN in the environment (a pod) or in Colab secrets'
    return t
api = HfApi(token=hf_token())
pre = PRESET.model.name
cks = sorted(f for f in api.list_repo_files(TRAINING_REPO) if f.startswith(f'{pre}/checkpoints/') and f.endswith('.safetensors') and '/fp8/' not in f)
assert cks, 'no checkpoint shipped yet'
last = cks[-1]
sd = load_file(hf_hub_download(TRAINING_REPO, last, token=hf_token()))
arm = {k: v for k, v in sd.items() if k.startswith(('lookup_arms.', 'lookup_norms.'))}
n = sum(v.numel() for v in arm.values())
out = f"{pre}/lookup_arm/{last.rsplit('/', 1)[-1].replace('.safetensors', '_arm_only.safetensors')}"
api.upload_file(path_or_fileobj=io.BytesIO(save(arm)), path_in_repo=out, repo_id=TRAINING_REPO, repo_type='model',
                commit_message=f'{pre}: the lookup arm alone from {last.rsplit("/", 1)[-1]} ({n/1e6:.2f}M parameters)')
print(f'shipped {out}: {len(arm)} tensors, {n/1e6:.2f}M parameters (from {last})')

### Notes
- **Sites.** `SITES` are block indices whose OUTPUT the arm reads; the default puts two arms early (the lookup circuit
  needs depth: a previous-token read feeding an induction read) and two deeper. A second run with other sites is a new
  craft name (`mini-beatrix-3-la<sites>`).
- **Memory.** The backward runs through the frozen trunk from the deepest loss to the shallowest arm, so activation memory is
  close to a full step's; 8 × 8 fits an 80 GB card with margin (C2 prints the peak).
- **What the arm cannot do yet.** Decode (`prefill`/sampling) refuses a model with arms; training and the forward reads
  are what this round needs. The reads (the battery and the probes, reports/byte_battery_2026-10-10/tools on the training
  repo) run on the shipped checkpoints and set the pass marks above.
- **Provenance.** The manifest records the source checkpoint, the fresh arm keys, the cursor (the trunk's 245,674 steps done,
  the arm phase active) and the frozen trunk; reports land under `reports/lookup_arm/`.